# PageRank Project - Ranking similarity between papers
### **Student: Aleksandra Zografska (48924A)**
### **Subject: Algorithms for Massive Datasets**


#### 0. Prep environment
- Download Spark
- Align cloudpickle version that caused errors
- Import Spark into notebook

In [2]:
!apt-get update -qq
!apt-get install openjdk-8-jdk-headless
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"

!pip install cloudpickle==2.2.0

import cloudpickle, shutil, os

new_cp_path = os.path.dirname(cloudpickle.__file__)
pyspark_cp_path = "/content/spark-3.1.1-bin-hadoop3.2/python/pyspark/cloudpickle"

print("Replacing:", pyspark_cp_path)
print("With:", new_cp_path)

shutil.copytree(new_cp_path, pyspark_cp_path, dirs_exist_ok=True)

print("Done!")

import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better

sc = spark.sparkContext

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libxtst6 openjdk-8-jre-headless
Suggested packages:
  openjdk-8-demo openjdk-8-source libnss-mdns fonts-dejavu-extra fonts-nanum
  fonts-ipafont-gothic fonts-ipafont-mincho fonts-wqy-microhei
  fonts-wqy-zenhei fonts-indic
The following NEW packages will be installed:
  libxtst6 openjdk-8-jdk-headless openjdk-8-jre-headless
0 upgraded, 3 newly installed, 0 to remove and 104 not upgraded.
Need to get 39.7 MB of archives.
After this operation, 144 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libxtst6 amd64 2:1.2.3-1build4 [13.4 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 op


#### 1. Download and Load Dataset into memory
Using https://www.kaggle.com/datasets/Cornell-University/arxiv

In [3]:
import os
os.environ['KAGGLE_USERNAME'] = "aleksandrazografska"
os.environ['KAGGLE_KEY'] = "KGAT_190e0a2b50e5230eee516795d4851ce5"
!kaggle datasets download -d Cornell-University/arxiv


Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0
 99% 1.56G/1.58G [00:14<00:00, 249MB/s]
100% 1.58G/1.58G [00:14<00:00, 114MB/s]


In [4]:
import zipfile

with zipfile.ZipFile("arxiv.zip", "r") as zip_ref:
    zip_ref.extractall("arxiv_data")

A regular json read didn't suffice. Session crached because of using too much ram, we need to load it by chunks.





In [5]:
import pandas as pd
chunks = pd.read_json(
    "arxiv_data/arxiv-metadata-oai-snapshot.json",
    lines=True,
    chunksize=10000
)

for chunk in chunks:
    print(chunk.head())
    break  # remove break to process full dataset

         id           submitter  \
0  704.0001      Pavel Nadolsky   
1  704.0002        Louis Theran   
2  704.0003         Hongjun Pan   
3  704.0004        David Callan   
4  704.0005  Alberto Torchinsky   

                                             authors  \
0  C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...   
1                    Ileana Streinu and Louis Theran   
2                                        Hongjun Pan   
3                                       David Callan   
4           Wael Abu-Shammala and Alberto Torchinsky   

                                               title  \
0  Calculation of prompt diphoton production cros...   
1           Sparsity-certifying Graph Decompositions   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                  comments  \
0  37 pages, 15 figures; published version   
1    To appear in Graph

In [6]:
''' As we will often print probability values,
we will modify the default setting of numpy so that
float entries in arrays are printed showing only a few decimal digits.'''
import numpy as np
np.set_printoptions(precision=3)


Helper functions for creating graph, getting the connection matrix and calculating a simple page rank.

In [7]:
def get_graph(pages, links):
    g = nx.DiGraph()

    for p in pages:
        g.add_node(p)

    for (a, b) in links:
        g.add_edge(pages[a], pages[b])

    return g

def get_connection_matrix(pages, links):
    incidency = {}
    for u in range(len(pages)):
        incidency[u] = []

    for (a, b) in links:
        incidency[a].append(b)

    connection_matrix = []
    for a in incidency:
       for b in incidency[a]:
            connection_matrix.append((b, a, 1./len(incidency[a])))

    return connection_matrix

def get_page_rank(pages, links,
                  verbose=False, tolerance=10e-7, max_iterations=100):
    connection_matrix = get_connection_matrix(pages, links)
    links_RDD = sc.parallelize(connection_matrix).cache()

    n = len(pages)
    page_rank = np.ones(n)/n
    old_page_rank = np.ones(n)

    iteration = 0
    dist = l2distance(old_page_rank, page_rank)
    while dist >= tolerance and \
          iteration < max_iterations:
        old_page_rank = page_rank
        page_rank_values = (links_RDD
                            .map(lambda t: (t[0], t[2]*page_rank[t[1]]))
                            .reduceByKey(lambda a, b: a+b)
                            .sortByKey()
                            .collect()
                           )
        page_rank = np.array([c for (i, c) in page_rank_values])

        if verbose:
            nice_print(page_rank, dist)

        iteration += 1
        dist = l2distance(old_page_rank, page_rank)

    print('{} iterations'.format(iteration))

    return page_rank

#### Page Rank: The author that collaborates the most with other authors

In [ ]:
from itertools import combinations

pages_set = set()
links_set = set()

for chunk in chunks:
    for authors in chunk["authors_parsed"]:
        names = [f"{a[1]} {a[0]}".strip() for a in authors]
        for name in names:
            pages_set.add(name)
        for a, b in combinations(names, 2):
            links_set.add((a, b))
            links_set.add((b, a))  # undirected — add both directions

# Convert to indexed lists (PageRank expects integer indices)
pages = list(pages_set)
page_index = {name: i for i, name in enumerate(pages)}
links = [(page_index[a], page_index[b]) for a, b in links_set]

# Now run PageRank
rank = get_page_rank(pages, links, verbose=True)

# See top authors
ranked = sorted(zip(pages, rank), key=lambda x: -x[1])
for author, score in ranked[:20]:
    print(f"{score:.6f}  {author}")

#### Visualize Simple Rank

In [ ]:

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

g = get_graph(pages, links)
# Labels: node -> author name
page_labels = {i: pages[i] for i in range(len(pages))}

# Layout
pos = nx.spring_layout(g, seed=42)

# node_color expects a list aligned to g.nodes() order
page_rank = [rank[i] for i in g.nodes()]

# Draw
opts = {"node_size": 200, "font_size": 8, "alpha": 0.8}
plt.figure(figsize=(14, 10))
nx.draw(g, with_labels=True, labels=page_labels, pos=pos,
        node_color=page_rank, cmap=plt.cm.coolwarm, **opts)
plt.colorbar(plt.cm.ScalarMappable(cmap=plt.cm.coolwarm), label="PageRank")
plt.show()
